# mini-ddpm: MNIST from scratch

Author: Fazal Hameed.

This clean notebook carries runnable source and commands, with no invented execution outputs. The recorded experiment uses a free T4 for speed; the CLI defaults to CPU. Read `PROTOCOL.md` for the fixed training and evaluation plan. The model has 550,465 parameters.


In [ ]:
import os,sys,subprocess,json
from pathlib import Path
import torch
WORK=Path('/content/mini_ddpm')
WORK.mkdir(exist_ok=True)
subprocess.check_call([sys.executable,'-m','pip','install','-q','numpy','Pillow','matplotlib'])
SOURCES={'mini_ddpm.py': '"""Small DDPM for MNIST. Equations follow Ho et al., arXiv:2006.11239.\n\nAll diffusion equations and the U-Net are implemented here using basic PyTorch.\nNo diffusion library, pretrained generator or pretrained evaluator is used.\n"""\nimport argparse\nimport copy\nimport csv\nimport gzip\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport shutil\nimport struct\nimport time\nimport urllib.request\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\n\nROOT=Path(__file__).resolve().parent\nRESOURCES={\n \'train-images-idx3-ubyte.gz\':\'f68b3c2dcbeaaa9fbdd348bbdeb94873\',\n \'train-labels-idx1-ubyte.gz\':\'d53e105ee54ea40749a09fcbcd1e9432\',\n \'t10k-images-idx3-ubyte.gz\':\'9fb629c4189551a2d022fa330f9573f3\',\n \'t10k-labels-idx1-ubyte.gz\':\'ec29112dd5afa0611ce80d1b7f02629c\',\n}\n\ndef sha256(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\ndef write_json(path,obj):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    path.write_text(json.dumps(obj,indent=2),encoding=\'utf-8\')\n\ndef atomic_save(obj,path):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    temp=path.with_suffix(path.suffix+\'.tmp\')\n    torch.save(obj,temp);os.replace(temp,path)\n\ndef backup(path):\n    """Optional verified backup, limited to the explicitly configured project."""\n    folder=os.environ.get(\'MINIDDPM_BACKUP\')\n    if not folder:return None\n    path=Path(path);relative=path.resolve().relative_to(ROOT.resolve())\n    target=Path(folder)/relative;target.parent.mkdir(parents=True,exist_ok=True)\n    expected=sha256(path)\n    temp=target.with_name(target.name+\'.copying\')\n    with path.open(\'rb\') as source,temp.open(\'wb\') as destination:\n        shutil.copyfileobj(source,destination)\n        destination.flush();os.fsync(destination.fileno())\n    assert sha256(temp)==expected,\'Backup checksum mismatch\'\n    os.replace(temp,target)\n    return {\'path\':str(relative),\'bytes\':path.stat().st_size,\'sha256\':expected}\n\ndef load_mnist(data_root):\n    data_root=Path(data_root);data_root.mkdir(parents=True,exist_ok=True)\n    arrays={}\n    for name,md5 in RESOURCES.items():\n        target=data_root/name\n        if not target.exists():\n            urllib.request.urlretrieve(\'https://ossci-datasets.s3.amazonaws.com/mnist/\'+name,target)\n        assert hashlib.md5(target.read_bytes()).hexdigest()==md5,name\n        raw=gzip.decompress(target.read_bytes())\n        magic,count=struct.unpack(\'>II\',raw[:8])\n        if magic==2051:\n            height,width=struct.unpack(\'>II\',raw[8:16]);assert (height,width)==(28,28)\n            array=np.frombuffer(raw,dtype=np.uint8,offset=16).copy().reshape(count,1,28,28)\n        else:\n            assert magic==2049\n            array=np.frombuffer(raw,dtype=np.uint8,offset=8).copy()\n        arrays[name]=torch.from_numpy(array)\n    return (arrays[\'train-images-idx3-ubyte.gz\'],arrays[\'train-labels-idx1-ubyte.gz\'].long(),\n            arrays[\'t10k-images-idx3-ubyte.gz\'],arrays[\'t10k-labels-idx1-ubyte.gz\'].long())\n\nclass TimeEmbedding(nn.Module):\n    def __init__(self,dim=128):\n        super().__init__();self.dim=dim\n        self.mlp=nn.Sequential(nn.Linear(dim,dim),nn.SiLU(),nn.Linear(dim,dim))\n    def forward(self,t):\n        frequencies=torch.exp(-math.log(10000)*torch.arange(self.dim//2,device=t.device)/(self.dim//2-1))\n        phase=t.float()[:,None]*frequencies[None,:]\n        return self.mlp(torch.cat((phase.sin(),phase.cos()),dim=1))\n\nclass ResidualBlock(nn.Module):\n    def __init__(self,inputs,outputs,time_dim=128):\n        super().__init__()\n        self.norm1=nn.GroupNorm(8,inputs);self.conv1=nn.Conv2d(inputs,outputs,3,padding=1)\n        self.time=nn.Linear(time_dim,outputs)\n        self.norm2=nn.GroupNorm(8,outputs);self.conv2=nn.Conv2d(outputs,outputs,3,padding=1)\n        self.skip=nn.Identity() if inputs==outputs else nn.Conv2d(inputs,outputs,1)\n    def forward(self,x,t):\n        h=self.conv1(F.silu(self.norm1(x)))+self.time(F.silu(t))[:,:,None,None]\n        return self.skip(x)+self.conv2(F.silu(self.norm2(h)))\n\nclass SmallUNet(nn.Module):\n    """28 -> 14 -> 7 -> 14 -> 28, time-conditioned residual/skip U-Net."""\n    def __init__(self,base=32):\n        super().__init__();self.base=base\n        self.time=TimeEmbedding(128);self.input=nn.Conv2d(1,base,3,padding=1)\n        self.enc1=ResidualBlock(base,base)\n        self.down1=nn.Conv2d(base,base,3,stride=2,padding=1)\n        self.enc2=ResidualBlock(base,base*2)\n        self.down2=nn.Conv2d(base*2,base*2,3,stride=2,padding=1)\n        self.mid1=ResidualBlock(base*2,base*2);self.mid2=ResidualBlock(base*2,base*2)\n        self.up2=nn.Conv2d(base*2,base*2,3,padding=1)\n        self.dec2=ResidualBlock(base*4,base*2)\n        self.up1=nn.Conv2d(base*2,base,3,padding=1)\n        self.dec1=ResidualBlock(base*2,base)\n        self.norm=nn.GroupNorm(8,base);self.output=nn.Conv2d(base,1,3,padding=1)\n    def forward(self,x,t):\n        temb=self.time(t)\n        skip1=self.enc1(self.input(x),temb)\n        skip2=self.enc2(self.down1(skip1),temb)\n        h=self.mid2(self.mid1(self.down2(skip2),temb),temb)\n        h=self.dec2(torch.cat((self.up2(F.interpolate(h,size=skip2.shape[-2:],mode=\'nearest\')),skip2),1),temb)\n        h=self.dec1(torch.cat((self.up1(F.interpolate(h,size=skip1.shape[-2:],mode=\'nearest\')),skip1),1),temb)\n        return self.output(F.silu(self.norm(h)))\n\nclass Diffusion:\n    def __init__(self,timesteps=1000,device=\'cpu\'):\n        self.timesteps=timesteps\n        self.betas=torch.linspace(1e-4,.02,timesteps,dtype=torch.float64,device=device)\n        self.alphas=1-self.betas\n        self.alpha_bars=torch.cumprod(self.alphas,dim=0)\n    def q_sample(self,x0,t,noise):\n        """Closed-form one-draw sample from q(x_t | x_0), paper Eq. 4."""\n        abar=self.alpha_bars[t].to(x0.dtype).view(-1,1,1,1)\n        return abar.sqrt()*x0+(1-abar).sqrt()*noise\n    def coefficients(self,t,s):\n        """Exact forward posterior q(x_s | x_t,x_0), allowing skipped indices.\n\n        s=-1 denotes clean x_0 under the zero-based implementation convention.\n        For s=t-1 these are the original DDPM posterior, Eq. 7.\n        """\n        at=self.alpha_bars[t]\n        a_s=self.alpha_bars[s] if s>=0 else at.new_tensor(1.)\n        effective_beta=1-at/a_s\n        variance=(1-a_s)/(1-at)*effective_beta\n        c0=a_s.sqrt()*effective_beta/(1-at)\n        ct=(at/a_s).sqrt()*(1-a_s)/(1-at)\n        return c0.float(),ct.float(),variance.clamp_min(0).float()\n    def indices(self,steps):\n        if not 2<=steps<=self.timesteps:raise ValueError(\'Sampling steps must be between 2 and T.\')\n        selected=torch.linspace(0,self.timesteps-1,steps,dtype=torch.float64).round().long().tolist()\n        assert len(set(selected))==steps and selected[0]==0 and selected[-1]==self.timesteps-1\n        return selected[::-1]\n    @torch.inference_mode()\n    def sample(self,model,count,steps,seed,method=\'ddpm\',amp=False):\n        device=next(model.parameters()).device\n        generator=torch.Generator(device=device).manual_seed(seed)\n        x=torch.randn((count,1,28,28),generator=generator,device=device)\n        indices=self.indices(steps)\n        for index,t in enumerate(indices):\n            s=indices[index+1] if index+1<len(indices) else -1\n            times=torch.full((count,),t,device=device,dtype=torch.long)\n            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp and device.type==\'cuda\'):\n                epsilon=model(x,times)\n            at=self.alpha_bars[t].float()\n            predicted_x0=((x-(1-at).sqrt()*epsilon.float())/at.sqrt()).clamp(-1,1)\n            if method==\'ddpm\':\n                c0,ct,var=self.coefficients(t,s)\n                x=c0*predicted_x0+ct*x\n                if s>=0:x=x+var.sqrt()*torch.randn(x.shape,generator=generator,device=device)\n            elif method==\'ddim\':\n                a_s=self.alpha_bars[s].float() if s>=0 else at.new_tensor(1.)\n                # Recompute epsilon after clipping x0, keeping the decomposition consistent.\n                eps=(x-at.sqrt()*predicted_x0)/(1-at).sqrt()\n                x=a_s.sqrt()*predicted_x0+(1-a_s).sqrt()*eps\n            else:raise ValueError(method)\n        return x.clamp(-1,1)\n\ndef image_grid(images,path,columns=8):\n    values=images.detach().cpu()\n    if values.dtype!=torch.uint8:values=((values.clamp(-1,1)+1)*127.5).round().to(torch.uint8)\n    count=len(values);rows=math.ceil(count/columns);gap=2\n    grid=np.zeros((rows*28+(rows+1)*gap,columns*28+(columns+1)*gap),dtype=np.uint8)\n    for i,value in enumerate(values):\n        row,col=divmod(i,columns);y=gap+row*(28+gap);x=gap+col*(28+gap)\n        grid[y:y+28,x:x+28]=value[0].numpy()\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);Image.fromarray(grid).save(path)\n\ndef synchronize(device):\n    if device.type==\'cuda\':torch.cuda.synchronize()\n\ndef environment(device):\n    return {\'python\':platform.python_version(),\'torch\':torch.__version__,\'numpy\':np.__version__,\n        \'device\':str(device),\'hardware\':torch.cuda.get_device_name(0) if device.type==\'cuda\' else platform.processor(),\n        \'cuda\':torch.version.cuda,\'threads\':torch.get_num_threads(),\'platform\':platform.platform()}\n\n@torch.inference_mode()\ndef validation_mse(model,images,diffusion,amp):\n    model.eval();generator=torch.Generator(device=images.device).manual_seed(9001)\n    total=0.\n    for batch in images.split(256):\n        t=torch.randint(diffusion.timesteps,(len(batch),),generator=generator,device=batch.device)\n        noise=torch.randn(batch.shape,generator=generator,device=batch.device)\n        noisy=diffusion.q_sample(batch,t,noise)\n        with torch.autocast(device_type=batch.device.type,dtype=torch.float16,enabled=amp):prediction=model(noisy,t)\n        total+=float(F.mse_loss(prediction.float(),noise))*len(batch)\n    model.train();return total/len(images)\n\ndef train(args):\n    device=torch.device(args.device);amp=device.type==\'cuda\'\n    torch.manual_seed(args.seed);torch.set_num_threads(args.threads)\n    if amp:torch.cuda.manual_seed_all(args.seed);torch.backends.cudnn.benchmark=True\n    raw,_,_,_=load_mnist(ROOT/\'data\')\n    split=torch.randperm(len(raw),generator=torch.Generator().manual_seed(31415))\n    train_ids,val_ids=split[:55000],split[55000:]\n    images=raw.to(device,dtype=torch.float32)/127.5-1\n    train_images,val_images=images[train_ids.to(device)],images[val_ids.to(device)]\n    model=SmallUNet(args.base).to(device);ema=copy.deepcopy(model).eval()\n    for parameter in ema.parameters():parameter.requires_grad_(False)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=args.lr,weight_decay=1e-4)\n    scaler=torch.amp.GradScaler(\'cuda\',init_scale=1024,enabled=amp)\n    diffusion=Diffusion(device=device)\n    config={\'epochs\':args.epochs,\'batch\':args.batch,\'seed\':args.seed,\'base\':args.base,\'lr\':args.lr,\n        \'timesteps\':1000,\'beta_start\':1e-4,\'beta_end\':.02,\'ema_decay\':.995,\n        \'train_images\':55000,\'validation_images\':5000,\'split_seed\':31415,\n        \'normalization\':\'uint8 / 127.5 - 1\',\'loss\':\'uniform-t epsilon-prediction MSE\',\n        \'source_sha256\':sha256(__file__),\'protocol_sha256\':sha256(ROOT/\'PROTOCOL.md\'),\n        \'data_checksums\':RESOURCES,\'parameters\':sum(p.numel() for p in model.parameters()),\n        \'environment\':environment(device),\'linear_convolution_precision\':\'float16 autocast with scaler\' if amp else \'float32\'}\n    assert config[\'parameters\']<=5_000_000\n    out=ROOT/\'runs/main\';out.mkdir(parents=True,exist_ok=True)\n    checkpoint=out/\'resume.pt\';start=0;history=[];train_seconds=0.;updates=0\n    generator=torch.Generator().manual_seed(args.seed+1)\n    if checkpoint.exists():\n        state=torch.load(checkpoint,map_location=\'cpu\',weights_only=False)\n        assert state[\'config\']==config,\'Existing run differs; use a fresh project directory.\'\n        model.load_state_dict(state[\'model\']);ema.load_state_dict(state[\'ema\'])\n        optimizer.load_state_dict(state[\'optimizer\']);scaler.load_state_dict(state[\'scaler\'])\n        start=state[\'epoch\'];history=state[\'history\'];train_seconds=state[\'train_seconds\'];updates=state[\'updates\']\n        generator.set_state(state[\'data_rng\']);torch.set_rng_state(state[\'cpu_rng\'])\n        if amp:torch.cuda.set_rng_state_all(state[\'cuda_rng\'])\n    write_json(out/\'config.json\',config)\n    print(\'TRAIN_START\',json.dumps({\'parameters\':config[\'parameters\'],\'environment\':config[\'environment\'],\'start_epoch\':start}),flush=True)\n    active_start=time.perf_counter()\n    for epoch in range(start,args.epochs):\n        model.train();order=torch.randperm(len(train_images),generator=generator).to(device)\n        total=0.;seen=0;synchronize(device);epoch_start=time.perf_counter()\n        for ids in order.split(args.batch):\n            x0=train_images[ids];t=torch.randint(1000,(len(ids),),device=device);noise=torch.randn_like(x0)\n            xt=diffusion.q_sample(x0,t,noise)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp):prediction=model(xt,t)\n            loss=F.mse_loss(prediction.float(),noise)\n            assert torch.isfinite(loss),\'Non-finite loss\'\n            scaler.scale(loss).backward();scaler.unscale_(optimizer)\n            norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n            assert torch.isfinite(norm),\'Non-finite gradient\'\n            old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update()\n            assert scaler.get_scale()>=old_scale,\'An optimizer update was skipped; investigate precision.\'\n            with torch.no_grad():\n                for target,source in zip(ema.parameters(),model.parameters()):target.lerp_(source,1-.995)\n            total+=float(loss.detach())*len(ids);seen+=len(ids);updates+=1\n        synchronize(device);seconds=time.perf_counter()-epoch_start;train_seconds+=seconds\n        row={\'epoch\':epoch+1,\'updates\':updates,\'training_mse\':total/seen,\n            \'validation_mse_ema\':validation_mse(ema,val_images,diffusion,amp),\n            \'epoch_training_seconds\':seconds,\'cumulative_training_seconds\':train_seconds}\n        history.append(row);print(\'EPOCH\',json.dumps(row),flush=True)\n        state={\'config\':config,\'epoch\':epoch+1,\'model\':model.state_dict(),\'ema\':ema.state_dict(),\n            \'optimizer\':optimizer.state_dict(),\'scaler\':scaler.state_dict(),\'history\':history,\n            \'train_seconds\':train_seconds,\'updates\':updates,\'data_rng\':generator.get_state(),\n            \'cpu_rng\':torch.get_rng_state(),\'cuda_rng\':torch.cuda.get_rng_state_all() if amp else None}\n        atomic_save(state,checkpoint);write_json(out/\'history.json\',history)\n        if (epoch+1)%3==0 or epoch+1==args.epochs:\n            backup(checkpoint);backup(out/\'config.json\');backup(out/\'history.json\')\n    weights={\'model\':{k:v.detach().cpu() for k,v in ema.state_dict().items()},\'config\':config,\n             \'epoch\':args.epochs,\'updates\':updates,\'weights\':\'EMA decay 0.995\'}\n    atomic_save(weights,out/\'model_ema.pt\');backup(out/\'model_ema.pt\')\n    result={\'config\':config,\'history\':history,\'updates\':updates,\'training_seconds\':train_seconds,\n        \'active_fit_wall_seconds_this_invocation\':time.perf_counter()-active_start,\n        \'weights_sha256\':sha256(out/\'model_ema.pt\'),\'training_completed\':True}\n    write_json(out/\'training_result.json\',result);backup(out/\'training_result.json\')\n    print(\'TRAIN_COMPLETE\',json.dumps({\'updates\':updates,\'training_seconds\':train_seconds,\'weights_sha256\':result[\'weights_sha256\']}),flush=True)\n\nclass DigitClassifier(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.conv=nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),\n            nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))\n        self.features=nn.Sequential(nn.Flatten(),nn.Linear(32*7*7,64),nn.ReLU())\n        self.head=nn.Linear(64,10)\n    def forward(self,x,features=False):\n        h=self.features(self.conv(x));return h if features else self.head(h)\n\ndef train_classifier(device):\n    torch.manual_seed(2718)\n    raw,y,test_raw,test_y=load_mnist(ROOT/\'data\')\n    x=raw.to(device,dtype=torch.float32)/127.5-1;y=y.to(device)\n    test_x=test_raw.to(device,dtype=torch.float32)/127.5-1;test_y=test_y.to(device)\n    model=DigitClassifier().to(device);optimizer=torch.optim.Adam(model.parameters(),lr=.001)\n    rows=[]\n    for epoch in range(3):\n        model.train();order=torch.randperm(len(x),device=device);total=0.\n        for ids in order.split(256):\n            optimizer.zero_grad(set_to_none=True);loss=F.cross_entropy(model(x[ids]),y[ids]);loss.backward();optimizer.step()\n            total+=float(loss.detach())*len(ids)\n        rows.append({\'epoch\':epoch+1,\'training_cross_entropy\':total/len(x)})\n    model.eval()\n    with torch.inference_mode():\n        correct=sum(int((model(b).argmax(1)==label).sum()) for b,label in zip(test_x.split(512),test_y.split(512)))\n    result={\'epochs\':3,\'seed\':2718,\'train_images\':60000,\'test_images\':10000,\'test_accuracy\':correct/10000,\n        \'features\':\'64-dimensional penultimate ReLU; not an ImageNet Inception FID\',\'history\':rows}\n    atomic_save({\'model\':model.cpu().state_dict(),\'result\':result},ROOT/\'runs/classifier.pt\');model.to(device)\n    write_json(ROOT/\'runs/classifier_result.json\',result)\n    print(\'CLASSIFIER_COMPLETE\',json.dumps(result),flush=True)\n    return model,test_x\n\n@torch.inference_mode()\ndef feature_statistics(classifier,images):\n    values=[];probabilities=[]\n    for batch in images.split(256):\n        feature=classifier(batch,features=True);values.append(feature.cpu().double())\n        probabilities.append(classifier.head(feature).softmax(1).cpu())\n    values=torch.cat(values);probabilities=torch.cat(probabilities)\n    mean=values.mean(0);centered=values-mean\n    covariance=centered.T@centered/(len(values)-1)\n    return mean,covariance,probabilities\n\ndef feature_distance(stats1,stats2):\n    m1,c1=stats1[:2];m2,c2=stats2[:2]\n    eig,vectors=torch.linalg.eigh((c1+c1.T)/2)\n    root=(vectors*eig.clamp_min(0).sqrt())@vectors.T\n    product=root@c2@root\n    cross=torch.linalg.eigvalsh((product+product.T)/2).clamp_min(0).sqrt().sum()\n    return float(((m1-m2).square().sum()+torch.trace(c1)+torch.trace(c2)-2*cross).clamp_min(0))\n\ndef evaluate(args):\n    device=torch.device(args.device);torch.set_num_threads(args.threads)\n    if device.type==\'cuda\':torch.backends.cudnn.benchmark=True\n    state=torch.load(ROOT/\'runs/main/model_ema.pt\',map_location=\'cpu\',weights_only=False)\n    assert state[\'config\'][\'source_sha256\']==sha256(__file__),\'Source changed after training\'\n    model=SmallUNet(state[\'config\'][\'base\']).to(device);model.load_state_dict(state[\'model\']);model.eval()\n    diffusion=Diffusion(device=device);amp=device.type==\'cuda\'\n    classifier,test_x=train_classifier(device)\n    order=torch.randperm(len(test_x),generator=torch.Generator().manual_seed(1618)).to(device)\n    reference=feature_statistics(classifier,test_x[order[:2048]])\n    calibration=feature_statistics(classifier,test_x[order[2048:2560]])\n    settings=[(\'ddpm\',1000),(\'ddpm\',200),(\'ddpm\',50),(\'ddim\',50)]\n    out=ROOT/\'results\';out.mkdir(exist_ok=True)\n    # Warm up convolution kernels, then rotate settings across three timing repeats.\n    diffusion.sample(model,64,5,9000,amp=amp);timings={f\'{method}_{steps}\':[] for method,steps in settings}\n    for repetition in range(3):\n        rotated=settings[repetition:]+settings[:repetition]\n        for method,steps in rotated:\n            synchronize(device);start=time.perf_counter()\n            diffusion.sample(model,64,steps,2026+repetition,method,amp)\n            synchronize(device);elapsed=time.perf_counter()-start\n            timings[f\'{method}_{steps}\'].append(elapsed)\n            print(\'TIMING\',json.dumps({\'repetition\':repetition,\'method\':method,\'steps\':steps,\'images\':64,\'seconds\':elapsed}),flush=True)\n    rows=[]\n    for method,steps in settings:\n        synchronize(device);start=time.perf_counter()\n        samples=diffusion.sample(model,512,steps,2026,method,amp)\n        synchronize(device);sample_seconds=time.perf_counter()-start\n        pixels=((samples+1)*127.5).round().clamp(0,255).to(torch.uint8).cpu()\n        # Metrics use exactly the persisted 8-bit samples, making them reproducible.\n        stats=feature_statistics(classifier,pixels.to(device,dtype=torch.float32)/127.5-1)\n        counts=torch.bincount(stats[2].argmax(1),minlength=10)\n        distribution=counts.double()/len(pixels)\n        entropy=float(-(distribution[distribution>0]*distribution[distribution>0].log()).sum()/math.log(10))\n        row={\'method\':method,\'steps\':steps,\'samples\':512,\'seed\':2026,\n            \'grid_images\':64,\'grid_selection\':\'first 64 generated images, no filtering\',\n            \'seconds_512\':sample_seconds,\'timings_64_seconds\':timings[f\'{method}_{steps}\'],\n            \'median_64_seconds\':float(np.median(timings[f\'{method}_{steps}\'])),\n            \'classifier_feature_distance\':feature_distance(stats,reference),\n            \'mean_classifier_confidence\':float(stats[2].max(1).values.mean()),\n            \'confidence_at_least_0_9_fraction\':float((stats[2].max(1).values>=.9).float().mean()),\n            \'predicted_class_counts\':counts.tolist(),\'normalized_predicted_class_entropy\':entropy}\n        image_grid(pixels[:64],out/f\'grid_{method}_{steps}.png\')\n        np.savez_compressed(out/f\'samples_{method}_{steps}.npz\',images=pixels.numpy())\n        rows.append(row);write_json(out/\'comparison_partial.json\',rows)\n        print(\'COMPARISON\',json.dumps(row),flush=True)\n    raw,_,_,_=load_mnist(ROOT/\'data\')\n    originals=raw[:8].to(device,dtype=torch.float32)/127.5-1\n    noise=torch.randn(originals.shape,generator=torch.Generator(device=device).manual_seed(808),device=device)\n    noisy=[]\n    for t in (0,49,199,499,999):\n        times=torch.full((8,),t,device=device,dtype=torch.long);noisy.append(diffusion.q_sample(originals,times,noise).cpu())\n    image_grid(torch.cat([originals.cpu()]+noisy),out/\'forward_noising.png\',columns=8)\n    result={\'rows\':rows,\'environment\':environment(device),\'checkpoint_sha256\':sha256(ROOT/\'runs/main/model_ema.pt\'),\n        \'sampling\':\'same trained EMA weights, full-range uniformly spaced original time indices, clipped predicted x0, fixed-small posterior variance\',\n        \'timing_protocol\':\'three rotated-order repeats, batch 64, CUDA synchronized, includes RNG and denoising; excludes model loading, plotting, metrics and file I/O\',\n        \'quality_protocol\':\'512 fixed-seed unfiltered samples per method; 2048 MNIST-test reference images; 8-bit images persisted; diagnostic classifier features, not standard FID\',\n        \'real_reference_calibration_distance\':feature_distance(calibration,reference),\n        \'classifier_result\':json.loads((ROOT/\'runs/classifier_result.json\').read_text())}\n    write_json(out/\'comparison.json\',result)\n    with (out/\'comparison.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as handle:\n        columns=[\'method\',\'steps\',\'median_64_seconds\',\'seconds_512\',\'classifier_feature_distance\',\'mean_classifier_confidence\',\'normalized_predicted_class_entropy\']\n        writer=csv.DictWriter(handle,fieldnames=columns);writer.writeheader()\n        writer.writerows({k:r[k] for k in columns} for r in rows)\n    records=[]\n    for folder in (ROOT/\'results\',ROOT/\'runs\'):\n        for path in sorted(folder.rglob(\'*\')):\n            if path.is_file() and path.name!=\'resume.pt\':\n                record=backup(path)\n                if record:records.append(record)\n    write_json(ROOT/\'backup_verification.json\',records);backup(ROOT/\'backup_verification.json\')\n    print(\'EVALUATION_COMPLETE\',out/\'comparison.json\',flush=True)\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'command\',choices=[\'train\',\'evaluate\'])\n    parser.add_argument(\'--device\',default=\'cpu\',choices=[\'cpu\',\'cuda\'])\n    parser.add_argument(\'--epochs\',type=int,default=12)\n    parser.add_argument(\'--batch\',type=int,default=256)\n    parser.add_argument(\'--base\',type=int,default=32)\n    parser.add_argument(\'--lr\',type=float,default=2e-4)\n    parser.add_argument(\'--seed\',type=int,default=42)\n    parser.add_argument(\'--threads\',type=int,default=min(4,os.cpu_count() or 1))\n    args=parser.parse_args()\n    if args.command==\'train\':train(args)\n    else:evaluate(args)\n\nif __name__==\'__main__\':main()\n', 'test_ddpm.py': '"""Check diffusion equations and computational contracts, independently of quality."""\nimport json\nimport math\nimport unittest\nfrom pathlib import Path\nimport torch\nfrom torch import nn\nfrom mini_ddpm import Diffusion,SmallUNet,feature_distance\n\nclass Tests(unittest.TestCase):\n    def setUp(self):torch.manual_seed(123);torch.set_num_threads(2)\n    def test_closed_form_noising(self):\n        d=Diffusion();x=torch.randn(3,1,28,28);z=torch.randn_like(x);t=torch.tensor([0,499,999])\n        a=d.alpha_bars[t].float().view(-1,1,1,1)\n        torch.testing.assert_close(d.q_sample(x,t,z),a.sqrt()*x+(1-a).sqrt()*z)\n        self.assertLess(float(d.alpha_bars[-1]),.0001)\n    def test_posterior_matches_original_adjacent_formula(self):\n        d=Diffusion()\n        for t in (1,10,500,999):\n            c0,ct,var=d.coefficients(t,t-1);a=d.alpha_bars[t];s=d.alpha_bars[t-1];beta=d.betas[t]\n            self.assertAlmostEqual(float(c0),float(s.sqrt()*beta/(1-a)),places=6)\n            self.assertAlmostEqual(float(ct),float(d.alphas[t].sqrt()*(1-s)/(1-a)),places=6)\n            self.assertAlmostEqual(float(var),float(beta*(1-s)/(1-a)),places=6)\n    def test_final_transition_has_no_noise(self):\n        for t in (0,49,999):\n            c0,ct,var=Diffusion().coefficients(t,-1)\n            self.assertAlmostEqual(float(c0),1.);self.assertEqual(float(ct),0.);self.assertEqual(float(var),0.)\n    def test_respacing_preserves_forward_marginals(self):\n        d=Diffusion()\n        for steps in (1000,200,50):\n            indices=d.indices(steps);self.assertEqual(len(indices),steps)\n            product=torch.tensor(1.,dtype=torch.float64)\n            previous=torch.tensor(1.,dtype=torch.float64)\n            for t in indices[::-1]:\n                product*=d.alpha_bars[t]/previous\n                torch.testing.assert_close(product,d.alpha_bars[t],rtol=1e-12,atol=1e-12)\n                previous=d.alpha_bars[t]\n    def test_unet_shape_parameters_and_cpu_gradient(self):\n        model=SmallUNet();self.assertLessEqual(sum(p.numel() for p in model.parameters()),5_000_000)\n        x=torch.randn(2,1,28,28);y=model(x,torch.tensor([0,999]));self.assertEqual(x.shape,y.shape)\n        y.square().mean().backward()\n        self.assertTrue(all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters()))\n    def test_sampler_counts_and_determinism(self):\n        class Zero(nn.Module):\n            def __init__(self):super().__init__();self.anchor=nn.Parameter(torch.zeros(()));self.calls=0\n            def forward(self,x,t):self.calls+=1;return torch.zeros_like(x)+self.anchor\n        d=Diffusion()\n        for steps in (1000,200,50):\n            model=Zero();a=d.sample(model,2,steps,88);self.assertEqual(model.calls,steps)\n            b=d.sample(model,2,steps,88);torch.testing.assert_close(a,b,rtol=0,atol=0)\n            self.assertTrue(torch.isfinite(a).all());self.assertTrue((a.abs()<=1).all())\n        model=Zero();a=d.sample(model,2,50,11,\'ddim\');b=d.sample(model,2,50,11,\'ddim\')\n        torch.testing.assert_close(a,b,rtol=0,atol=0)\n    def test_feature_distance_identity_and_symmetry(self):\n        a=(torch.tensor([1.,2.],dtype=torch.float64),torch.diag(torch.tensor([2.,3.],dtype=torch.float64)))\n        b=(torch.tensor([2.,4.],dtype=torch.float64),torch.diag(torch.tensor([4.,1.],dtype=torch.float64)))\n        self.assertAlmostEqual(feature_distance(a,a),0.,places=7)\n        expected=5+(math.sqrt(2)-2)**2+(math.sqrt(3)-1)**2\n        self.assertAlmostEqual(feature_distance(a,b),expected,places=7)\n        self.assertAlmostEqual(feature_distance(a,b),feature_distance(b,a),places=7)\n\nif __name__==\'__main__\':\n    suite=unittest.defaultTestLoader.loadTestsFromTestCase(Tests)\n    result=unittest.TextTestRunner(verbosity=2).run(suite)\n    Path(\'tests_result.json\').write_text(json.dumps({\'tests\':result.testsRun,\'failures\':len(result.failures),\'errors\':len(result.errors),\'passed\':result.wasSuccessful()},indent=2))\n    raise SystemExit(0 if result.wasSuccessful() else 1)\n', 'sample.py': '"""Generate an 8x8 grid from the supplied trained EMA checkpoint, without MNIST."""\nimport argparse\nfrom pathlib import Path\nimport time\nimport torch\nfrom mini_ddpm import SmallUNet,Diffusion,image_grid,synchronize\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--checkpoint\',default=\'runs/main/model_ema.pt\')\n    parser.add_argument(\'--steps\',type=int,default=200)\n    parser.add_argument(\'--method\',choices=[\'ddpm\',\'ddim\'],default=\'ddpm\')\n    parser.add_argument(\'--device\',choices=[\'cpu\',\'cuda\'],default=\'cpu\')\n    parser.add_argument(\'--threads\',type=int,default=4)\n    parser.add_argument(\'--seed\',type=int,default=2026)\n    parser.add_argument(\'--count\',type=int,default=64)\n    parser.add_argument(\'--output\',default=\'samples.png\')\n    args=parser.parse_args();torch.set_num_threads(args.threads)\n    device=torch.device(args.device)\n    checkpoint=torch.load(args.checkpoint,map_location=\'cpu\',weights_only=False)\n    model=SmallUNet(checkpoint[\'config\'][\'base\']).to(device)\n    model.load_state_dict(checkpoint[\'model\']);model.eval()\n    diffusion=Diffusion(device=device)\n    synchronize(device);start=time.perf_counter()\n    images=diffusion.sample(model,args.count,args.steps,args.seed,args.method,device.type==\'cuda\')\n    synchronize(device);seconds=time.perf_counter()-start\n    image_grid(images,args.output)\n    print(f\'Saved {args.count} samples to {Path(args.output).resolve()} in {seconds:.3f} seconds on {device}.\')\n\nif __name__==\'__main__\':main()\n', 'analyze.py': '"""Create figures from saved measured histories; never generate assumed scores."""\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\n\nROOT=Path(__file__).resolve().parent\ndef main():\n    result=json.loads((ROOT/\'runs/main/training_result.json\').read_text())\n    comparison=json.loads((ROOT/\'results/comparison.json\').read_text())\n    assert result[\'training_completed\'] and len(result[\'history\'])==12\n    assert {(r[\'method\'],r[\'steps\']) for r in comparison[\'rows\']}=={(\'ddpm\',1000),(\'ddpm\',200),(\'ddpm\',50),(\'ddim\',50)}\n    history=result[\'history\']\n    plt.rcParams.update({\'font.size\':10,\'axes.spines.top\':False,\'axes.spines.right\':False})\n    fig,ax=plt.subplots(figsize=(6,3.2))\n    ax.plot([r[\'epoch\'] for r in history],[r[\'training_mse\'] for r in history],marker=\'o\',label=\'Training (online weights)\',color=\'#1d5a78\')\n    ax.plot([r[\'epoch\'] for r in history],[r[\'validation_mse_ema\'] for r in history],marker=\'s\',label=\'Validation (EMA weights)\',color=\'#cf6b2e\')\n    ax.set(xlabel=\'Epoch\',ylabel=\'Noise-prediction MSE\',title=\'Recorded loss curves\')\n    ax.legend(fontsize=8);ax.grid(alpha=.2);fig.tight_layout()\n    fig.savefig(ROOT/\'results/loss_curves.png\',dpi=200);plt.close(fig)\n    rows=comparison[\'rows\'];baseline=next(r for r in rows if r[\'method\']==\'ddpm\' and r[\'steps\']==1000)\n    for row in rows:row[\'speedup_vs_ddpm1000\']=baseline[\'median_64_seconds\']/row[\'median_64_seconds\']\n    summary={\'parameters\':result[\'config\'][\'parameters\'],\'epochs\':12,\'updates\':result[\'updates\'],\n        \'training_seconds\':result[\'training_seconds\'],\'environment\':result[\'config\'][\'environment\'],\n        \'final_training_mse\':history[-1][\'training_mse\'],\'final_validation_mse_ema\':history[-1][\'validation_mse_ema\'],\n        \'classifier_test_accuracy\':comparison[\'classifier_result\'][\'test_accuracy\'],\n        \'real_reference_calibration_distance\':comparison[\'real_reference_calibration_distance\'],\'rows\':rows}\n    (ROOT/\'results/summary.json\').write_text(json.dumps(summary,indent=2),encoding=\'utf-8\')\n    print(json.dumps(summary,indent=2))\n\nif __name__==\'__main__\':main()\n', 'evaluate_saved.py': '"""Repeat comparisons using the already trained, fixed evaluator checkpoint."""\nfrom types import SimpleNamespace\nimport argparse\nimport json\nimport torch\nimport mini_ddpm as experiment\n\ndef restore_classifier(device):\n    path=experiment.ROOT/\'runs/classifier.pt\'\n    state=torch.load(path,map_location=\'cpu\',weights_only=False)\n    assert state[\'result\'][\'epochs\']==3 and state[\'result\'][\'seed\']==2718\n    assert state[\'result\']==json.loads((experiment.ROOT/\'runs/classifier_result.json\').read_text())\n    model=experiment.DigitClassifier().to(device);model.load_state_dict(state[\'model\']);model.eval()\n    _,_,test_raw,_=experiment.load_mnist(experiment.ROOT/\'data\')\n    test_x=test_raw.to(device,dtype=torch.float32)/127.5-1\n    print(\'RESTORED_FIXED_CLASSIFIER\',state[\'result\'][\'test_accuracy\'],flush=True)\n    return model,test_x\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--device\',default=\'cpu\',choices=[\'cpu\',\'cuda\'])\n    parser.add_argument(\'--threads\',type=int,default=4)\n    args=parser.parse_args()\n    experiment.train_classifier=restore_classifier\n    experiment.evaluate(args)\n', 'PROTOCOL.md': '# mini-ddpm experimental plan\n\nAuthor: Fazal Hameed. Written before training on 18 October 2026.\n\n- Train an unconditional, time-conditioned small U-Net from random initialization.\n  Use 28x28 MNIST, input scaling to [-1,1], a 1,000-step linear beta schedule\n  from 0.0001 to 0.02, and uniformly sampled timestep noise-prediction MSE.\n- A fixed seed-31415 split uses 55,000 of the 60,000 training images for fitting\n  and 5,000 for validation. Do not fit the diffusion model on the official test set.\n- Use seed 42, base width 32, batch 256, AdamW learning rate 0.0002, weight\n  decay 0.0001, gradient clipping 1.0, and EMA decay 0.995. Train for 12 epochs.\n  Use the final EMA checkpoint, without selecting epochs using sample quality.\n- Run on an available free Colab T4 for speed. The program defaults to CPU and\n  uses FP32 on CPU. Record actual hardware, precision and measured runtime;\n  GPU timing does not establish laptop CPU runtime.\n- Compare DDPM ancestral sampling with 1,000, 200 and 50 model evaluations using\n  the same weights and the entire original noise range. For shorter chains,\n  preserve original alpha-bar marginals and use the exact respaced Gaussian\n  posterior; never just stop a 1,000-step chain early or reuse adjacent-step\n  coefficients when skipping timesteps.\n- Bonus: implement deterministic DDIM with 50 evaluations using the same weights.\n- Sample 512 images for each setting with seed 2026. Show the first 64 as an\n  unfiltered 8x8 grid. Save the complete 8-bit batches for metric reproduction.\n- Measure batch-64 sampling in three repeats, rotating execution order. Warm up\n  kernels first, synchronize CUDA around timing, and exclude plotting/file I/O.\n- Train a separate small digit classifier from scratch on all 60,000 original\n  training labels for three epochs (seed 2718). Report its accuracy on all\n  10,000 test images. Use its 64-dimensional features for an explicitly\n  nonstandard MNIST feature-distance diagnostic, with a fixed 2,048-image test\n  reference (seed 1618). Include a separate real-512 vs reference calibration.\n  Classifier confidence and predicted digit coverage are diagnostics, not proof\n  of sample correctness or substitutes for visual inspection. Do not call this\n  metric standard Inception FID or claim MNIST classifier confidence is accuracy\n  on unlabeled generated digits.\n- Retain full loss history, configuration, source and data checksums, weights,\n  sample batches, timing repetitions and evaluator weights. Report actual\n  problems and limitations, including one training seed and finite sample size.\n- References: DDPM https://arxiv.org/abs/2006.11239 ;\n  DDIM https://arxiv.org/abs/2010.02502 ;\n  respaced DDPM reference https://github.com/openai/improved-diffusion/blob/main/improved_diffusion/respace.py ;\n  MNIST mirror/checksums https://github.com/pytorch/vision/blob/main/torchvision/datasets/mnist.py .\n', 'README.md': '# mini-ddpm: MNIST diffusion from scratch\n\nAuthor: Fazal Hameed.\n\nThis project implements the forward diffusion schedule, direct noising,\ntime-conditioned small U-Net, noise-prediction MSE, ancestral DDPM sampling,\ncorrect posterior respacing, and the optional deterministic DDIM sampler using\nbasic PyTorch. It uses no pretrained generator or diffusion framework.\n\nThe program defaults to CPU. The recorded experiments use a free Colab T4 for\nspeed; GPU times must not be presented as laptop CPU times. The U-Net has 550,465\nparameters, below the five-million limit.\n\n## Install and verify\n\nUse Python 3.10 or newer. From this folder:\n\n```text\npython -m pip install -r requirements.txt\npython test_ddpm.py\n```\n\nThe tests check the noising equation, adjacent and respaced posterior formulas,\nthe clean final transition, exact model-evaluation counts, seeded sampling,\nfinite CPU gradients and classifier feature-distance calculations. They do not\nestablish image quality.\n\n## Generate samples with the included weights\n\nNo dataset download is needed for these commands:\n\n```text\npython sample.py --device cpu --method ddpm --steps 1000 --output ddpm_1000.png\npython sample.py --device cpu --method ddpm --steps 200 --output ddpm_200.png\npython sample.py --device cpu --method ddpm --steps 50 --output ddpm_50.png\npython sample.py --device cpu --method ddim --steps 50 --output ddim_50.png\n```\n\nEach command writes an 8x8 grid. CUDA is optional: replace `--device cpu` with\n`--device cuda` when supported. CPU speed depends on hardware and thread count.\nThe default sampling seed is 2026; no samples are filtered or selected by quality.\n\n## Reproduce training and comparisons\n\nUse a fresh project folder for a new run. MNIST is downloaded from the mirror\nused by Torchvision, and all four original gzip MD5 checksums are verified.\n\n```text\npython mini_ddpm.py train --device cpu --threads 4\npython mini_ddpm.py evaluate --device cpu --threads 4\npython analyze.py\n```\n\nOn Colab, use `--device cuda --threads 2` for training and evaluation. The clean\nnotebook contains the exact source and commands. Defaults match the recorded\n12-epoch, seed-42, batch-256 experiment. An interrupted training command resumes\nfrom the last completed epoch only when its configuration, environment and\nsource hashes match. An unfinished epoch is repeated. Resumed wall times are\nrecorded per invocation, while epoch training times are accumulated. Bitwise\nequivalence across CPU/GPU or CUDA library versions is not guaranteed.\n\n`evaluate` fits the fixed three-epoch digit classifier from scratch, evaluates\nits test accuracy, runs three rotated-order timing repetitions, generates 512\nunfiltered images per sampling setting and writes grids/metrics. Run this in the\nmatching source version. The optional `MINIDDPM_BACKUP` environment variable\nenables verified project-file backups to a mounted Drive directory.\n\n## Read the evidence\n\n- `PROTOCOL.md`: plan written before training.\n- `runs/main/config.json`, `history.json`, `training_result.json`: setup and losses.\n- `runs/main/model_ema.pt`: final EMA generator weights and configuration.\n- `runs/classifier.pt`, `classifier_result.json`: evaluator weights and accuracy.\n- `results/grid_ddpm_1000.png`, `grid_ddpm_200.png`, `grid_ddpm_50.png`: required grids.\n- `results/grid_ddim_50.png`: optional bonus grid.\n- `results/samples_*.npz`: all 512 generated 8-bit images per setting.\n- `results/comparison.json`, `comparison.csv`, `summary.json`: measured comparisons.\n- `results/loss_curves.png`, `forward_noising.png`: method/training figures.\n- `output/pdf/Mini_DDPM_Report_Fazal_Hameed.pdf`: three-page experimental report.\n\nThe respaced samplers cover the original 1,000-step noise range. The experiment\ndoes not train separate 200- or 50-timestep generators. The 64-dimensional MNIST\nclassifier feature distance is a diagnostic, not standard Inception FID.\nClassifier confidence on unlabeled generated images is not their accuracy.\nOne generator training seed and 512 samples per setting limit general conclusions.\n\n## References and data\n\n- Ho, Jain and Abbeel, Denoising Diffusion Probabilistic Models (2020):\n  https://arxiv.org/abs/2006.11239 .\n- Song, Meng and Ermon, Denoising Diffusion Implicit Models (2020):\n  https://arxiv.org/abs/2010.02502 .\n- Respaced posterior reference:\n  https://github.com/openai/improved-diffusion/blob/main/improved_diffusion/respace.py .\n- MNIST: LeCun, Cortes and Burges. The original IDX image/label files are fetched\n  from https://ossci-datasets.s3.amazonaws.com/mnist/ using checksums documented\n  in https://github.com/pytorch/vision/blob/main/torchvision/datasets/mnist.py .\n  Raw dataset files are not redistributed in the submission ZIP.\n', 'requirements.txt': 'torch>=2.8\nnumpy>=1.26\nPillow>=10\nmatplotlib>=3.8\n'}
for name,text in SOURCES.items():(WORK/name).write_bytes(text.encode('utf-8'))
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Device:',DEVICE, 'PyTorch:',torch.__version__)
def run(command):
    process=subprocess.Popen(command,cwd=WORK,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:print(line,end='',flush=True)
    assert process.wait()==0,'Command failed; inspect output.'


In [ ]:
run([sys.executable,'test_ddpm.py'])


## Train and compare

Fixed 12 epochs, seed 42, 55,000 training images and 5,000 validation images. Sampling uses the final EMA checkpoint. The evaluator is trained separately from scratch. Free GPU availability varies; CPU execution can take substantially longer.


In [ ]:
run([sys.executable,'mini_ddpm.py','train','--device',DEVICE,'--threads','2'])
run([sys.executable,'mini_ddpm.py','evaluate','--device',DEVICE,'--threads','2'])
run([sys.executable,'analyze.py'])


In [ ]:
from IPython.display import display,Image
display(Image(filename=str(WORK/'results/loss_curves.png')))
for name in ['grid_ddpm_1000.png','grid_ddpm_200.png','grid_ddpm_50.png','grid_ddim_50.png']:
    print(name)
    display(Image(filename=str(WORK/'results'/name)))
print((WORK/'results/comparison.csv').read_text())


## Preserve results

Colab runtime files are temporary. Download the model and evidence before disconnecting. The recorded submission includes the checked three-page report; its Windows PDF typesetter is separate from training and is not needed to run the experiment.


In [ ]:
import zipfile
from google.colab import files
archive=Path('/content/mini_ddpm_results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(WORK.rglob('*')):
        if path.is_file() and not any(p in ('data','__pycache__') for p in path.relative_to(WORK).parts) and path.name!='resume.pt':z.write(path,path.relative_to(WORK))
files.download(str(archive))
